# Evaluative Synthesis, How Good Are Our Models Across All Three Points

## Table Of Contents

- [1. What This Synthesis Covers And How To Read It](#sec1)
- [2. Point One, Anomaly Detection, In Summary](#sec2)
- [3. Point Two, Conversion Risk, How Good Is The Hazard Model](#sec3)
- [4. Point Three, Decision Support, How Good Are The Policies](#sec4)
- [5. One Name Meaning Different Things, The RISK_SCORE Divergence](#sec5)
- [6. A Ceiling Worth Naming, Where More Did Not Help](#sec6)
- [7. So How Good Are Our Models, An Honest Verdict](#sec7)
- [8. What Would Make This Stronger, Concrete Next Steps](#sec8)

<a id="sec1"></a>

## 1. What This Synthesis Covers And How To Read It

This notebook exists because Pelle asked directly, once the discrete time
hazard model existed and the cost model had grown to read a genuine multi
horizon forecast rather than one snapshot, what can actually be said
about this project's results, and how good the models are across their
various aspects. That question deserves a direct, honest answer in one
place rather than an implicit one scattered across eight other notebooks
and two planning documents.

Every number below is read from the same shared files every other
notebook in this project already writes to, results/decision_support/approach_comparison.csv
for point three's own three approach groups plus point two's own hazard
model rows inside that same file, results/hazard_model/hazard_estimates.csv
and hazard_survival_model.ipynb for point two, and Giorgio's and Leo's own
saved result files for point one and the RUL side exploration covered in
section 6 below. Nothing here is recomputed from scratch, so a reader can
check any claim against its own source rather than taking this notebook's
word for it.

Point one and the RUL exploration are not this notebook's own work, and
are summarized here as faithfully as possible from Giorgio's and Leo's
own saved metrics rather than rechecked or rerun. Point two and point
three are Pelle's own responsibility and are checked directly against
their own source notebooks and result files. This notebook sits alongside
comparison.ipynb rather than replacing it, comparison.ipynb answers what
point three's own numbers say, section by section, in detail, while this
notebook asks the broader question across the whole project, and is
willing to name plainly what still falls short of finished.

In [1]:
import os
import pandas as pd

pd.set_option('display.width', 160)

results_path = os.path.join('..', '..', 'results', 'decision_support')
comparison_path = os.path.join(results_path, 'approach_comparison.csv')
assert os.path.exists(comparison_path), (
    'approach_comparison.csv not found. Run every approach_*.ipynb '
    'notebook, every temporal_window_*.ipynb notebook, and '
    'hazard_survival_model.ipynb at least once first, each one appends '
    'its own rows to this shared file.'
)
results = pd.read_csv(comparison_path)
test_results = results[results.split == 'test'].copy()
print(f'{len(results)} rows loaded, {results.approach_group.nunique()} approach groups, '
      f'{results.approach.nunique()} approaches, train and test split each')


34 rows loaded, 4 approach groups, 17 approaches, train and test split each


<a id="sec2"></a>

## 2. Point One, Anomaly Detection, In Summary

Point one is Giorgio's own work, not rechecked here, read directly from
gio/method_a_pca_density/metrics.txt and gio/method_b_autoencoder_hi/metrics.txt
below. Method A fits PCA and a Hotelling T2 plus Q control chart on 464
patients, cross sectional, one score per patient, no visit history.
Method B trains an autoencoder on 3471 CN visits from 823 patients, then
scores 9922 visits across 2093 patients longitudinally, one health index
value, HI, per visit.

**Read the per visit and per patient numbers below as two different
questions, not a contradiction.** Method B's own per visit numbers are
inflated by pseudo replication, a patient with many visits contributes
many correlated rows to the same area under the ROC curve calculation,
exactly the same subject level independence concern this project's own
cost model and fairness audit are careful about elsewhere. Method B's own
metrics file already corrects for this with a second, per patient,
last visit only version, the one worth trusting more.

In [2]:
gio_path = os.path.join('..', '..', 'notebooks', 'anomaly_detection')
method_a_metrics = os.path.join(gio_path, 'method_a_pca_density', 'metrics.txt')
method_b_metrics = os.path.join(gio_path, 'method_b_autoencoder_hi', 'metrics.txt')

for label, path in [('Method A, PCA plus Hotelling T2 and Q', method_a_metrics),
                     ('Method B, autoencoder health index', method_b_metrics)]:
    print(f'=== {label} ===')
    with open(path) as f:
        print(f.read())


=== Method A, PCA plus Hotelling T2 and Q ===
n_patients = 464  (CN=269, MCI=161, AD=34)
n_components retained = 14 (explains 91.9% variance)
T2 UCL (99%) = 31.67, Q UCL (99%) = 4.59

ROC-AUC, CN vs MCI (combined T2+Q score): 0.696
ROC-AUC, CN vs AD (combined T2+Q score): 0.890
ROC-AUC, CN vs (MCI+AD combined): 0.730

=== Method B, autoencoder health index ===
n_visits = 9922, n_patients = 2093
CN visits used to fit autoencoder: 3471 (823 patients)
Autoencoder: 16-6-16 MLP, n_iter=648, final train loss=0.1516

-- AUC using every visit (per-visit DIAGNOSIS_LABEL as target) --
NOTE: patients contribute multiple correlated visits here, so
this inflates apparent sample size (pseudo-replication). See the
last-visit-only AUC below for a per-patient, non-repeated check.
  CN vs MCI: 0.644  (n=7928)
  CN vs AD: 0.783  (n=5465)
  CN vs MCI+AD: 0.687

-- AUC using only each patient's LAST visit (one row per patient) --
  CN vs MCI: 0.627  (n=1408)
  CN vs AD: 0.777  (n=1364)
  CN vs MCI+AD: 0.70

Method A's own raw discrimination numbers read higher than Method B's,
CN vs MCI 0.696 against 0.627, CN vs AD 0.890 against 0.777, CN vs MCI
plus AD 0.730 against 0.700, despite Method A being the simpler, cross
sectional method. The two methods are not scored on identical patient
sets, 464 for Method A against 1364 to 1408 for Method B's own last
visit only numbers, so this reads as a directional observation rather
than a controlled comparison, worth someone on the team checking on a
shared patient set at some point rather than a settled ranking between
the two methods.

Method B's own real strength is not this discrimination number but its
slope, the health index's own rate of change per year, which rises
cleanly with severity, median 0.0039 per year in CN, 0.0082 in MCI,
0.0320 in AD, roughly double from CN to MCI and roughly eight times from
CN to AD. A single snapshot AUC would not show this, a genuinely clean
longitudinal degradation signal that reads as this project's strongest
result from point one, and one point two's own hazard model does not yet
use directly, an anomaly score and a health index slope both remain
candidate input features for a future version of the hazard model, named
already in IMPLEMENTATION_PLAN.md's own section four.

<a id="sec3"></a>

## 3. Point Two, Conversion Risk, How Good Is The Hazard Model

hazard_survival_model.ipynb fits four candidate estimators, a Logistic
Regression and a Random Forest, each on two feature tiers, a core tier of
seven features every biomarker modality contributes to, and an extended
tier of nine features that adds the cerebrospinal fluid markers on top,
following 05 pm, lesson 6, Survival Analysis via Neural Models, for the
discrete time hazard framing itself.

In [3]:
hazard_group = test_results[test_results.approach_group == 'hazard_model'].copy()
hazard_group = hazard_group.sort_values('approach')
cols = ['approach', 'n_rows', 'auc', 'auc_ci_low', 'auc_ci_high', 'notes']
print(hazard_group[cols].to_string(index=False))
print()

attribution_group = test_results[test_results.approach_group == 'attribution']
real_outcome = attribution_group[attribution_group.approach == 'real_outcome_attribution']
print('For reference, real_outcome_attribution, a one visit ahead classifier with no time axis:')
print(real_outcome[cols].to_string(index=False))


                approach  n_rows      auc  auc_ci_low  auc_ci_high                                                                                                  notes
      hazard_forest_core    2291 0.790876    0.758735     0.823957       Discrete time hazard estimator, forest, core feature tier, 7 features, following 05 pm lesson 6.
  hazard_forest_extended    1738 0.791233    0.755236     0.828907   Discrete time hazard estimator, forest, extended feature tier, 9 features, following 05 pm lesson 6.
    hazard_logistic_core    2291 0.743059    0.704754     0.782472     Discrete time hazard estimator, logistic, core feature tier, 7 features, following 05 pm lesson 6.
hazard_logistic_extended    1738 0.768142    0.728554     0.805919 Discrete time hazard estimator, logistic, extended feature tier, 9 features, following 05 pm lesson 6.

For reference, real_outcome_attribution, a one visit ahead classifier with no time axis:
                approach  n_rows      auc  auc_ci_low  auc_c

**Read the four confidence intervals above as overlapping, not as a clean
ranking.** Core logistic reaches test AUC 0.743 with a 95 percent interval
of 0.705 to 0.782, core forest reaches 0.791 with an interval of 0.759 to
0.824, extended logistic reaches 0.768 with an interval of 0.729 to
0.806, extended forest, the chosen estimator, reaches 0.791 with an
interval of 0.755 to 0.829. Every one of these intervals overlaps every
other one, so this project cannot say with real confidence that the
extended tier or the forest model is truly better than the simpler
alternatives on this sample, only that this sample cannot tell any of the
four apart with real confidence, a more honest reading than the single
point estimates alone would give.

The event being predicted, DIAGNOSIS_WORSENED_NEXT, is rare, an event
rate of about 5 to 6 percent depending on split and tier, printed inside
hazard_survival_model.ipynb itself. Discriminating a rare event is a
harder task than the AUC number alone suggests, worth keeping in mind
throughout this section.

**Read the coverage cost as a real trade, not a free upgrade.** The
extended tier's own richer feature set is not measured at every visit the
way structural MRI is, so it reaches only 1738 of the 2291 test rows the
core tier reaches, 24 percent fewer. Choosing the extended tier trades a
small, statistically uncertain accuracy gain against a real drop in how
many rows the model can score at all, which is exactly why every
downstream notebook keeps a fallback estimator, a plain Logistic
Regression fit on DIAGNOSIS and AMYLOID_STATUS categories alone, for the
rows the chosen tier cannot reach. That fallback is not a weak stand in,
it reaches its own test AUC of 0.676, printed inside every temporal_window
notebook's own section two, not far below the full model's 0.791.

**Read real_outcome_attribution as the genuinely independent comparison
point.** It answers a different question, one visit predicting only the
very next one, no time axis, on a different, smaller, more restrictive
sample of 620 test rows, following the same 8 biomarker levels the
attribution track uses. Its own test AUC, 0.619, with a 95 percent
interval of 0.546 to 0.705, sits below every one of the four hazard
estimators, and this project's own hazard estimators mostly clear that
entire interval rather than only its point estimate. That is a real,
checked sign that the discrete time, multi visit framing is adding
something a single snapshot classifier does not have, this section's own
strongest and best supported result.

<a id="sec4"></a>

## 4. Point Three, Decision Support, How Good Are The Policies

Point three's own three approach groups, interval policy, fairness
correction, and attribution, are read below the same way comparison.ipynb
reads them, as answers to different questions rather than one ranking.

In [4]:
interval_group = test_results[test_results.approach_group == 'interval_policy'].copy()
interval_group = interval_group.sort_values('total_cost')
cols = ['approach', 'n_rows', 'total_cost', 'catch_rate', 'didi', 'notes']
print(interval_group[cols].to_string(index=False))


             approach  n_rows   total_cost  catch_rate     didi                                                                                                                                                         notes
    snapshot_adaptive    3665 18387.063297   88.324873 7.199504                                                                             Predict then Optimize on the current visit RISK_SCORE only, menu 3, 6, 12 months.
  trajectory_adaptive    3665 18710.377293   87.309645 6.919439 Uses each patient own risk trajectory when available, 73.8 percent of test rows have a real trajectory, the rest fall back to a snapshot only recommendation.
pipeline_health_index    3665 21640.887371         NaN 2.029942                  Snapshot adaptive policy on RISK_SCORE blended 80/20 with Method B HI, matched by RID and nearest exam date within 180 days, 67.1% coverage.
         fixed_policy    3665 22479.361175   90.862944 0.000000                                                 

**Read fixed, snapshot adaptive, and trajectory adaptive together, the
three genuinely comparable policies, before forecast adaptive and the
pipeline blend, which are not on the same scale, covered in sections 5
and 6 below.** The two adaptive policies among these three, snapshot
adaptive and trajectory adaptive, each beat the fixed policy by around 17
to 18 percent on cost, 18387.1 or 18710.4 against 22479.4, while giving
up only 2 to 4 points of catch rate, 88.3 or 87.3 percent against 90.9
percent. That is a real, worthwhile trade for a clinic weighing routine
visit cost against a missed conversion, and this section's own clearest
positive result.

Snapshot adaptive and trajectory adaptive land within about 300 in cost
and one point of catch rate of each other, trajectory adaptive actually
costs slightly more and catches slightly fewer conversions despite
reading a patient's fuller visit history rather than only their current
one. That is not a mistake in either policy, it is a genuine finding
about how much a longer history is worth here, covered as this
document's second ceiling style pattern in section 6 below.

In [5]:
fairness_group = test_results[test_results.approach_group == 'fairness_correction'].copy()
cols = ['approach', 'n_rows', 'r2', 'r2_ci_low', 'r2_ci_high', 'didi', 'notes']
print(fairness_group[cols].to_string(index=False))


     approach  n_rows        r2  r2_ci_low  r2_ci_high     didi                                                                                         notes
unconstrained     753  0.056403        NaN         NaN 0.128358      No fairness constraint, this is the accuracy ceiling and DIDI floor for this comparison.
 fixed_lambda     753 -0.290264        NaN         NaN 0.048055                                  Fixed Lagrangian multiplier alpha=5.0, DIDI threshold 0.042.
  dual_lambda     753  0.067258        NaN         NaN 0.122326 Learned Lagrangian multiplier via gradient ascent, trained alpha 0.038, DIDI threshold 0.042.


**Read this as a trade off surface, not a single winner**, the same way
approach_fairness_correction.ipynb itself frames it, following 07 ciml,
lesson 2, Lagrangian Approaches for Constraint Injection. unconstrained
sets this comparison's own accuracy ceiling, test r2 0.056, no fairness
penalty holding it back, and applies no DIDI threshold at all, test DIDI
0.128. fixed_lambda buys a large DIDI drop, down to 0.048, at a
catastrophic accuracy cost, test r2 falls to negative 0.290, worse than
always guessing the test split's own average RISK_SCORE. dual_lambda, the
learned multiplier, protects accuracy well, test r2 0.067, at or slightly
above unconstrained, but only barely moves DIDI, to 0.122. That is an
honest, partial confirmation of the course lesson's own claim that a
learned multiplier keeps more accuracy than a fixed one for a similar
fairness level, the accuracy half holds up clearly here, the similar
fairness level half does not, dual lambda's own fairness gain over doing
nothing at all is small on this run.

In [6]:
attribution_group = test_results[test_results.approach_group == 'attribution'].copy()
cols = ['approach', 'n_rows', 'r2', 'r2_ci_low', 'r2_ci_high', 'auc', 'notes']
print(attribution_group[cols].to_string(index=False))


                  approach  n_rows       r2  r2_ci_low  r2_ci_high      auc                                                                                                                                                       notes
       decline_attribution     620 0.019909  -0.045140    0.037698      NaN                                                         Lasso on per month biomarker slopes rather than raw levels, alpha=0.01, test coverage 73.8 percent.
decline_attribution_forest     620 0.088353   0.007153    0.134700      NaN                                                            Random Forest on per month biomarker slopes, 200 trees, max depth 4, test coverage 73.8 percent.
  real_outcome_attribution     620      NaN        NaN         NaN 0.619036 L1 logistic regression predicting a real recorded DIAGNOSIS_WORSENED_NEXT event from 8 biomarker levels, scored by area under the ROC curve rather than r2.
                     lasso     753 0.414381   0.344180    0.466635      

**Read lasso, random_forest, and decline_attribution together on r2, and
real_outcome_attribution on its own AUC scale**, exactly as comparison.ipynb
already frames it. random_forest reaches the higher test r2 of the two
level based methods, 0.234 against lasso's 0.200, but its own train r2,
0.509, is more than double its test figure, a real sign of overfitting
even at a capped tree depth of four. decline_attribution, predicting each
patient's own per month biomarker slope rather than their current level,
finds essentially no signal, test r2 negative 0.022, an honest null
result worth reporting plainly rather than a disappointing one worth
hiding. real_outcome_attribution's own 0.619 test AUC is covered already
in section 3 above, the one attribution approach that answers a genuinely
different question, against a real recorded event rather than against
RISK_SCORE.

<a id="sec5"></a>

## 5. One Name Meaning Different Things, The RISK_SCORE Divergence

This is this document's own central cross cutting honest finding, stated
plainly rather than left implicit. RISK_SCORE no longer means one single
thing project wide.

In [7]:
import json

targets = {
    'temporal_window/1_fixed_policy.ipynb': 'fixed_policy',
    'temporal_window/2_snapshot_adaptive.ipynb': 'snapshot_adaptive',
    'temporal_window/3_trajectory_adaptive.ipynb': 'trajectory_adaptive',
    'temporal_window/4_forecast_adaptive.ipynb': 'forecast_adaptive',
    'risk_factors/1_global.ipynb': 'lasso and random_forest attribution',
    'risk_factors/2_decline.ipynb': 'decline_attribution',
    os.path.join('..', 'fairness', 'approach_fairness_correction.ipynb'): 'fairness correction, all three variants',
    os.path.join('..', 'pipeline', 'approach_pipeline_anomaly.ipynb'): 'pipeline_health_index',
}

for path, label in targets.items():
    with open(path) as f:
        nb = json.load(f)
    src = ''.join(''.join(c['source']) for c in nb['cells'] if c['cell_type'] == 'code')
    reads_real = 'HAZARD_ESTIMATE' in src and 'hazard_estimates.csv' in src
    reads_placeholder = 'placeholder_risk_score(' in src
    status = 'real HAZARD_ESTIMATE' if reads_real else ('still placeholder_risk_score' if reads_placeholder else 'reads neither, uses the real outcome directly')
    print(f'{label:42s} reads {status}')


fixed_policy                               reads real HAZARD_ESTIMATE
snapshot_adaptive                          reads real HAZARD_ESTIMATE
trajectory_adaptive                        reads real HAZARD_ESTIMATE
forecast_adaptive                          reads real HAZARD_ESTIMATE
lasso and random_forest attribution        reads real HAZARD_ESTIMATE
decline_attribution                        reads real HAZARD_ESTIMATE
fairness correction, all three variants    reads still placeholder_risk_score
pipeline_health_index                      reads still placeholder_risk_score


fixed_policy, snapshot_adaptive, trajectory_adaptive, and forecast_adaptive,
in notebooks/decision_support, were swapped over to point two's own real
HAZARD_ESTIMATE, described in full in NOTES.md's own swapping section.
approach_fairness_correction.ipynb, approach_pipeline_anomaly.ipynb, and
risk_factors/1_global.ipynb and risk_factors/2_decline.ipynb still call
util.placeholder_risk_score, a simple function built from DIAGNOSIS and
AMYLOID_STATUS categories alone, each one still carries its own SWAP
POINT comment in the code marking exactly where the real estimate belongs
once point two exists, which it now does, checked directly in the cell
above rather than assumed from memory.

The concrete consequence differs by track. The fairness correction
numbers in section 4 above describe how much accuracy a Lagrangian
penalty costs against the placeholder's own two column structure, not
against the real, richer hazard model, so whether the same trade off
holds once the swap happens is genuinely still open. The lasso and
random_forest attribution numbers describe which of eight biomarkers best
predict the placeholder, which is itself only a function of two
categorical fields, so a biomarker showing up as a strong predictor there
is at best a proxy finding through whatever correlates DIAGNOSIS and
AMYLOID_STATUS with those eight biomarkers, not a direct read on what
drives the real hazard model's own output. pipeline_health_index blends
Method B's HI with the placeholder rather than the real estimate, exactly
as that notebook's own section one text already says, a stand in built
before point two existed, not yet revisited now that it does.
real_outcome_attribution and decline_attribution are the exception,
neither reads RISK_SCORE at all, one against a real recorded event, one
against a patient's own biomarker slope, so both stay exactly as
informative as before regardless of this gap.

This is not being treated as a bug to rush and fix inside this synthesis
itself. Completing the swap in four more notebooks, then rerunning the
fairness and attribution comparisons against the real estimate, is real,
scoped follow up work of its own, named directly in section 8 below
rather than attempted here.

<a id="sec6"></a>

## 6. A Ceiling Worth Naming, Where More Did Not Help

This pattern is worth naming honestly because it showed up more than
once, in more than one part of the project, not as a single isolated
result.

The first instance is point two's own extended feature tier. Adding the
cerebrospinal fluid markers on top of the core tier moves the Random
Forest's own test AUC from 0.790876 to 0.791233, to three decimal places
0.791 either way, no meaningful change, and costs 24 percent of the test
rows their eligibility, 1738 against 2291, since CSF is not measured at
every visit the way structural MRI is, covered fully in section 3 above.

The second instance is trajectory_adaptive. Reading a patient's fuller
visit history rather than only their current one costs slightly more and
catches slightly fewer conversions than snapshot_adaptive on the test
split, 18710.4 against 18387.1 in cost, 87.3 percent against 88.3 percent
in catch rate, covered fully in section 4 above. More history did not
translate into a better recommendation here either.

In [8]:
leo_path = os.path.join('..', '..', 'notebooks', 'remaining_useful_life', 'rul_results.csv')
rul = pd.read_csv(leo_path)
print(rul.to_string(index=False))


         modalities  n_features  n_rows      MAE     RMSE  MAE_baseline
(demographics only)           3    2787 1.638856 1.958045      1.605256
                mri           6    2787 1.342001 1.626690      1.605256
                pet           5    2787 1.315824 1.736544      1.605256
                csf           6    2787 1.378480 1.763369      1.605256
        mri+pet+csf          11    2787 1.132810 1.469867      1.605256


**Read this table as the balancing case, included for honesty rather
than to fit a tidy narrative.** Leo's own Remaining Useful Life model,
predicting years to conversion rather than classifying it, shows mean
absolute error falling from 1.34 years with structural MRI alone, to 1.32
with amyloid PET alone, to 1.38 with CSF alone, to 1.13 with all three
modalities combined, a real, meaningful further gain from combining
modalities rather than a plateau, against a naive baseline error of 1.61.
Demographics alone, by contrast, land at 1.64, worse than the naive
baseline, the one modality here that does not help at all on its own.

The honest reading of the contrast is that whether more information helps
seems to depend on what is being asked of it, not on some general rule
about this dataset. A continuous target with a wide, informative range,
years of remaining useful life, apparently has more room for extra
modalities to sharpen an estimate than a rare, roughly 5 percent binary
event does once a model already has enough signal to rank patients
reasonably, or a recommendation menu with only three or four discrete
choices does once a policy already lands in the right neighborhood most
of the time. This project has not tested that explanation directly, it is
offered here as the most plausible reading of what the numbers above
actually show, not as a settled conclusion.

<a id="sec7"></a>

## 7. So How Good Are Our Models, An Honest Verdict

Organized by the three aspects that came up repeatedly across sections 2
through 6 above, ranking, calibration, and decision quality, rather than
by point number again.

On ranking, every model in this project that was checked against a real
outcome, point two's own four hazard estimators and real_outcome_attribution
together, discriminates meaningfully better than chance, and the hazard
model's own multi visit framing checkably beats a naive one visit
classifier, section 3's own comparison against real_outcome_attribution.
This is this project's strongest and best supported result.

On calibration, the one place this project checked it directly,
forecast_adaptive's own near universal shortest interval recommendation,
covered fully in temporal_window/4_forecast_adaptive.ipynb, NOTES.md, and
comparison.ipynb section 3, the honest answer is that this is genuinely
unresolved. An AUC of 0.791 says the model ranks patients sensibly, it
says nothing about whether one specific predicted probability can be
trusted at face value, and compounding that probability across several
future steps, exactly what a multi horizon forecast does, is precisely
where an uncalibrated model's weaknesses show up most.

On decision quality, the two genuinely comparable adaptive policies show
a real, honest improvement over the fixed schedule they are both
measured against, 17 to 18 percent cheaper for a small catch rate cost,
though the improvement from adaptivity itself levels off quickly,
snapshot adaptive already captures most of what trajectory adaptive
offers, section 6's own second ceiling
style finding.

On fairness, a real trade off exists and was measured rather than
assumed, though still against the placeholder score rather than the real
one, section 5 above, so this project's own fairness numbers describe a
trade off that is real in shape but not yet confirmed at the scale that
will matter once the real hazard model feeds this track too.

Taken together, this project's models are good at the thing checked most
carefully, telling higher risk patients apart from lower risk ones,
genuinely useful for the thing checked second most carefully, choosing a
cheaper monitoring schedule than a one size fits all default, and
honestly still unproven at the thing checked least, whether one specific
predicted probability or one specific fairness trade off can be trusted
at face value once every track reads the same real estimate.

<a id="sec8"></a>

## 8. What Would Make This Stronger, Concrete Next Steps

In roughly the order that would most increase confidence in the answer
section 7 above gives.

A real calibration check on hazard_survival_model.ipynb's own chosen
estimator, predicted probability against observed frequency within bins,
already named as an open item in IMPLEMENTATION_PLAN.md and NOTES.md, the
single check most likely to change how forecast_adaptive's own numbers in
section 4 above should be read.

Completing the RISK_SCORE swap in approach_fairness_correction.ipynb,
approach_pipeline_anomaly.ipynb, risk_factors/1_global.ipynb, and
risk_factors/2_decline.ipynb, then rerunning the fairness and attribution
comparisons against the real hazard model rather than the placeholder,
section 5 above, to check whether the accuracy fairness trade off and the
attribution rankings in section 4 hold up once they are measured against
the model this project actually built rather than a two column stand in.

Revisiting fixed_lambda's own collapse to a negative test r2, section 4
above, worth checking whether a smaller fixed multiplier than 5.0 finds a
less extreme point on the same trade off curve, since the current fixed
lambda result reads more like an example of how badly a poorly chosen
fixed multiplier can go than a fair comparison against the dual, learned
version.

A closer look at why decline_attribution finds no signal, section 4
above, whether a longer, less noisy slope window, or a target other than
a raw per month rate, changes that null result, or whether it genuinely
reflects that this patient population's biomarker trajectories do not
move in a straight enough line for a slope to be a meaningful summary on
their own.

This document itself only covers what has been checked so far. Naming
plainly what remains open is as much a part of an honest synthesis as
reporting what already looks solid, and is the note this document closes
on.